# 🥉 Bronze Layer - Raw Data Ingestion

This notebook ingests raw data from 5 distinct sources into the Bronze layer:

1. **US Accidents** - Kaggle/CSV format
2. **NYC 311 Service Requests** - Socrata API / CSV
3. **USGS Earthquake Hazards** - GeoJSON API
4. **OSM Infrastructure** - Overpass API (Hospitals/Fire Stations)
5. **US Neighborhoods** - Census Bureau GeoJSON

## Architecture Decision
This layer does MINIMAL transformation - just lands data in raw format.
All cleaning/parsing happens in Silver layer.

In [ ]:
# Import transformation functions from src module
import sys
sys.path.append('../src')

from bronze_ingestion import (
    Config,
    fetch_us_accidents,
    fetch_nyc_311_requests,
    fetch_usgs_earthquakes,
    fetch_osm_infrastructure,
    fetch_us_neighborhoods,
    run_bronze_ingestion,
    upload_to_bronze,
    get_minio_client
)

import os
import pandas as pd
from pathlib import Path

print("✅ Imports successful")

## Configuration

In [ ]:
# Display current configuration
config = Config()

print("=" * 60)
print("BRONZE LAYER CONFIGURATION")
print("=" * 60)
print(f"MinIO Endpoint: {config.MINIO_ENDPOINT}")
print(f"Bronze Bucket: {config.BRONZE_BUCKET}")
print(f"Local Data Dir: {config.BRONZE_BUCKET}")
print(f"USGS API: {config.USGS_API_URL[:50]}...")
print(f"OSM API: {config.OSM_API_URL[:50]}...")
print(f"NYC 311 API: {config.NYC311_API_URL[:50]}...")

## Step 1: Fetch US Accidents Data

In [ ]:
# Fetch US Accidents data
success = fetch_us_accidents()
print(f"US Accidents fetch: {'✅ SUCCESS' if success else '❌ FAILED'}")

# Verify the output
accidents_path = Config.LOCAL_DATA_DIR / "us_accidents" / "part-00000.parquet"
if accidents_path.exists():
    df = pd.read_parquet(accidents_path)
    print(f"\n📊 US Accidents Data:")
    print(f"   Rows: {len(df)}")
    print(f"   Columns: {list(df.columns)}")
    print(f"\n   Sample:")
    display(df.head(2))

## Step 2: Fetch NYC 311 Requests

In [ ]:
# Fetch NYC 311 data
success = fetch_nyc_311_requests()
print(f"NYC 311 fetch: {'✅ SUCCESS' if success else '❌ FAILED'}")

# Verify the output
nyc_path = Config.LOCAL_DATA_DIR / "nyc_311" / "part-00000.parquet"
if nyc_path.exists():
    df = pd.read_parquet(nyc_path)
    print(f"\n📊 NYC 311 Data:")
    print(f"   Rows: {len(df)}")
    print(f"   Columns: {list(df.columns)}")
    display(df.head(2))

## Step 3: Fetch USGS Earthquakes

In [ ]:
# Fetch USGS Earthquakes
success = fetch_usgs_earthquakes()
print(f"USGS Earthquakes fetch: {'✅ SUCCESS' if success else '❌ FAILED'}")

# Verify the output
usgs_path = Config.LOCAL_DATA_DIR / "usgs_earthquakes" / "part-00000.parquet"
if usgs_path.exists():
    df = pd.read_parquet(usgs_path)
    print(f"\n📊 USGS Earthquakes Data:")
    print(f"   Rows: {len(df)}")
    print(f"   Columns: {list(df.columns)}")
    display(df.head(2))

## Step 4: Fetch OSM Infrastructure

In [ ]:
# Fetch OSM Infrastructure
success = fetch_osm_infrastructure()
print(f"OSM Infrastructure fetch: {'✅ SUCCESS' if success else '❌ FAILED'}")

# Verify the output
osm_path = Config.LOCAL_DATA_DIR / "osm_infrastructure" / "part-00000.parquet"
if osm_path.exists():
    df = pd.read_parquet(osm_path)
    print(f"\n📊 OSM Infrastructure Data:")
    print(f"   Rows: {len(df)}")
    print(f"   Columns: {list(df.columns)}")
    display(df.head(2))

## Step 5: Fetch US Neighborhoods

In [ ]:
# Fetch US Neighborhoods
success = fetch_us_neighborhoods()
print(f"US Neighborhoods fetch: {'✅ SUCCESS' if success else '❌ FAILED'}")

# Verify the output
neighborhoods_path = Config.LOCAL_DATA_DIR / "us_neighborhoods" / "part-00000.parquet"
if neighborhoods_path.exists():
    df = pd.read_parquet(neighborhoods_path)
    print(f"\n📊 US Neighborhoods Data:")
    print(f"   Rows: {len(df)}")
    print(f"   Columns: {list(df.columns)}")
    display(df.head(2))

## Bronze Layer Summary

In [ ]:
# Summary of all ingested data
print("=" * 60)
print("BRONZE LAYER INGESTION SUMMARY")
print("=" * 60)

sources = [
    ("US Accidents", "us_accidents"),
    ("NYC 311", "nyc_311"),
    ("USGS Earthquakes", "usgs_earthquakes"),
    ("OSM Infrastructure", "osm_infrastructure"),
    ("US Neighborhoods", "us_neighborhoods")
]

total_rows = 0
for name, folder in sources:
    path = Config.LOCAL_DATA_DIR / folder / "part-00000.parquet"
    if path.exists():
        df = pd.read_parquet(path)
        rows = len(df)
        total_rows += rows
        print(f"✅ {name:25} | {rows:5} rows | {path}")
    else:
        print(f"❌ {name:25} | NOT FOUND")

print("-" * 60)
print(f"{'TOTAL':25} | {total_rows:5} rows")

## Upload to MinIO (Optional)

In [ ]:
# Upload to MinIO - requires proper network configuration
# client = get_minio_client()
# if client:
#     print("✅ MinIO client connected")
# else:
#     print("⚠️ MinIO not available - using local storage")

print("📁 Data is stored locally at:", Config.LOCAL_DATA_DIR)
print("\nFor MinIO upload, ensure proper Docker networking:")
print("  - Spark container must be on same network as MinIO")
print("  - Use internal Docker hostname (e.g., 'minio' not 'localhost')")

---

## Next Steps

Proceed to **Silver Layer** for:
- Spatial standardization (Apache Sedona)
- LLM enrichment (Ollama)
- Data quality improvements